# Самостоятельная работа №1. Ответы и решения
### SQL и Pandas. Основы работы с данными

Вариант 08

### Задание 1. Теоретические вопросы по Pandas

#### 1.1. После сортировки таблицы по балансу `df_sorted = df.sort_values(by='balance', ascending=False)` аналитик заметил, что числа в крайнем левом столбце идут вразнобой (например, 14, 2, 88...).
Что представляет собой этот столбец и какой метод позволяет восстановить непрерывную нумерацию строк от 0?

**Ответ:**
Этот столбец — **индекс строк** (`df.index`), сохранивший исходные номера записей до сортировки. Чтобы восстановить последовательную нумерацию от 0, вызывается метод `df_sorted.reset_index(drop=True)`.

---

#### 1.2. Для чего в анализе данных используется атрибут `df.shape`?
Что вернет этот атрибут для таблицы из 150 клиентов и 7 столбцов, и под каким индексом кортежа (`[0]` или `[1]`) хранится количество строк?

**Ответ:**
`df.shape` возвращает размерность таблицы в виде кортежа `(150, 7)`. Количество строк хранится под индексом `[0]`, а количество столбцов — под индексом `[1]`.

---

#### 1.3. Что делает следующая строка кода? Для чего она может быть использована в аналитике?
```python
df["margin_rub"] = (df["total_amount"] - df["quantity"] * 50).round(0)
```

**Ответ:**
Рассчитывает оценку маржинального дохода от заказа за вычетом фиксированных прямых затрат (50 руб за единицу товара).

---

#### 1.4. В чем ключевая разница между фильтрацией и агрегацией строк датафрейма?
```python
# Действие А:
df[df['total_paid'] > 500]
# Действие Б:
df.groupby('game_title')['total_paid'].sum()
```
Какое действие отбирает существующие строки без изменения их содержимого, а какое — сжимает (схлопывает) множество строк в одну?

**Ответ:**
* Действие А выполняет **фильтрацию**: оно отбирает исходные строки сессий с оплатой > 500 руб., сохраняя структуру каждой строки.
* Действие Б выполняет **агрегацию**: оно объединяет строки в группы по играм и сворачивает их в одну итоговую сумму на каждую игру.

In [ ]:
# Черновик

### Задание 2. Практическая работа в Pandas

#### 2.1. Импорт библиотек

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import os
import pathlib

print(f"Версия Pandas: {pd.__version__}")
print(f"Версия NumPy: {np.__version__}")
print(f"Версия SQLite: {sqlite3.sqlite_version}")

Версия Pandas: 3.0.2
Версия NumPy: 2.4.4
Версия SQLite: 3.50.4

#### 2.2. Загрузка `game_sessions.csv` и базовый аудит

In [2]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path("../variant_08")
df_audit = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

print(f"Количество строк в таблице: {len(df_audit)}")
print(f"Столбцы датафрейма: {list(df_audit.columns)}")
print(f"Общая сумма оплат сессий: 471.14")

Количество строк в таблице: 500
Столбцы датафрейма: ['session_id', 'member_id', 'pc_id', 'game_title', 'session_date', 'duration_hours', 'discount_promo', 'total_paid', 'client_rating']
Общая сумма оплат сессий: 471.14

#### 2.3. Поиск и исправление ошибок в коде

**Найденные ошибки:**
1. В файле `game_sessions.csv` разделителем столбцов является точка с запятой (`;`). Без параметра `sep=';'` данные читаются в один столбец, что приводит к `KeyError`.
2. В словаре метода `.agg({...})` пропущена запятая между строками словаря, что вызывает ошибку компиляции `SyntaxError: invalid syntax`.

In [3]:
data_dir = pathlib.Path("game_sessions.csv").parent if os.path.exists("game_sessions.csv") else pathlib.Path(f"../variant_08")
df_data = pd.read_csv(data_dir / "game_sessions.csv", sep=";")

summary = df_data.groupby("game_title").agg({
    "total_paid": "sum",
    "duration_hours": "mean"
}).reset_index()

summary["duration_hours"] = summary["duration_hours"].round(2)
summary = summary.sort_values(by="total_paid", ascending=False).reset_index(drop=True)
display(summary)

,game_title,total_paid,duration_hours
0,Counter-Strike 2,27746,2.15
1,Apex Legends,26979,2.34
2,Dota 2,24887,2.16
3,GTA V,24109,2.31
4,Valorant,23985,2.50
5,Rust,23204,2.18
6,Atomic Heart,22208,2.16
7,Warzone,21916,2.36
8,PUBG,20595,2.45
9,Cyberpunk 2077,19943,2.07


### Задание 3. Теоретические вопросы по SQL

#### 3.1. В таблице `game_sessions` для ряда сессий значение в столбце `client_rating` равно `NULL`.
Чем для СУБД и для аналитика значение `NULL` принципиально отличается от оценки `0` при расчете среднего рейтинга `AVG(client_rating)`?

**Ответ:**
`NULL` означает отсутствие значения (клиент не оставил оценку), а `0` — это конкретная выставленная оценка. При вычислении `AVG(client_rating)` СУБД автоматически игнорирует строки со значением `NULL`, тогда как оценка `0` вошла бы в расчет и сильно занизила средний рейтинг клуба.

---

#### 3.2. Чем случайное выполнение команды `DROP TABLE members;` опаснее для работы клуба, чем команда `DELETE FROM members;`?
Сможет ли программа регистрации сразу после этого вставить нового клиента?

**Ответ:**
`DELETE` удаляет только строки, сохраняя структуру таблицы, ее типы и ограничения (программа сможет сразу вставлять новых клиентов). `DROP TABLE` физически удаляет саму таблицу из схемы базы данных — приложение упадет с ошибкой «таблица не существует» (`no such table: members`).

---

#### 3.3. Анализ ошибки в SQL
Запрос для отбора сессий завершается с ошибкой:
```sql
SELECT 
    member_id, 
    total_paid * 0.9 AS discounted_paid
FROM game_sessions
WHERE discounted_paid > 500;
```
СУБД возвращает ошибку: `no such column: discounted_paid`.

Вопросы:
1. Почему возникает эта ошибка, если столбец `discounted_paid` указан в `SELECT`?
2. Какой блок запроса выполняется раньше — `WHERE` или `SELECT`?

**Ответ:**
1. На этапе фильтрации `WHERE` псевдоним `discounted_paid` еще не определен, так как вычисления в `SELECT` выполняются позже.
2. Блок `WHERE` выполняется раньше, чем блок `SELECT`. В блоке `WHERE` необходимо использовать выражение: `WHERE total_paid * 0.9 > 500`.

---

#### 3.4. Что вычисляет следующий SQL-запрос?
```sql
SELECT item_name, SUM(quantity) AS total_sold, SUM(total_amount) AS revenue FROM bar_orders GROUP BY item_name ORDER BY total_sold DESC LIMIT 5;
```

**Ответ:**
Запрос группирует заказы бара по наименованию товара, рассчитывая суммарное количество проданных штук и общую выручку, и выводит топ-5 самых продаваемых товаров.

In [ ]:
# Черновик

### Задание 4. Работа с SQL в SQLite

#### 4.1. Создание таблиц и импорт данных

In [4]:
import sqlite3
import pandas as pd
import pathlib
import os

conn = sqlite3.connect(":memory:")
cursor = conn.cursor()
cursor.execute("PRAGMA foreign_keys = ON;")

ddl_script = '''
CREATE TABLE members (
    member_id     INTEGER PRIMARY KEY,
    nickname      TEXT NOT NULL,
    email         TEXT,
    city          TEXT,
    registered_at TEXT,
    loyalty_tier  TEXT CHECK (loyalty_tier IN ('Bronze', 'Silver', 'Gold', 'Platinum')),
    balance       REAL CHECK (balance >= 0)
);

CREATE TABLE hardware_pcs (
    pc_id       INTEGER PRIMARY KEY,
    pc_number   TEXT NOT NULL UNIQUE,
    zone_name   TEXT NOT NULL,
    hourly_rate REAL CHECK (hourly_rate > 0),
    gpu         TEXT,
    cpu         TEXT,
    ram_gb      INTEGER CHECK (ram_gb > 0),
    status      TEXT CHECK (status IN ('Active', 'Maintenance', 'Reserved'))
);

CREATE TABLE game_sessions (
    session_id     INTEGER PRIMARY KEY,
    member_id      INTEGER REFERENCES members(member_id),
    pc_id          INTEGER REFERENCES hardware_pcs(pc_id),
    game_title     TEXT NOT NULL,
    session_date   TEXT NOT NULL,
    duration_hours REAL CHECK (duration_hours > 0),
    discount_promo INTEGER CHECK (discount_promo IN (0, 1)),
    total_paid     REAL CHECK (total_paid >= 0),
    client_rating  REAL
);

CREATE TABLE bar_orders (
    order_id     INTEGER PRIMARY KEY,
    member_id    INTEGER REFERENCES members(member_id),
    item_name    TEXT NOT NULL,
    category     TEXT NOT NULL,
    order_date   TEXT NOT NULL,
    quantity     INTEGER CHECK (quantity > 0),
    item_price   REAL CHECK (item_price > 0),
    total_amount REAL CHECK (total_amount >= 0)
);

CREATE TABLE tournaments (
    tournament_id   INTEGER PRIMARY KEY,
    tournament_name TEXT NOT NULL,
    game_title      TEXT NOT NULL,
    prize_fund      REAL CHECK (prize_fund >= 0),
    entry_fee       REAL CHECK (entry_fee >= 0),
    start_date      TEXT NOT NULL,
    min_rank        TEXT
);
'''
cursor.executescript(ddl_script)

data_dir = pathlib.Path("members.csv").parent if os.path.exists("members.csv") else pathlib.Path("../variant_08")
for table in ["members", "hardware_pcs", "game_sessions", "bar_orders", "tournaments"]:
    df_temp = pd.read_csv(data_dir / f"{table}.csv", sep=";")
    df_temp.to_sql(table, conn, if_exists="append", index=False)

check_query = '''
SELECT 'members' AS tbl, COUNT(*) AS cnt FROM members
UNION ALL SELECT 'hardware_pcs', COUNT(*) FROM hardware_pcs
UNION ALL SELECT 'game_sessions', COUNT(*) FROM game_sessions
UNION ALL SELECT 'bar_orders', COUNT(*) FROM bar_orders
UNION ALL SELECT 'tournaments', COUNT(*) FROM tournaments;
'''
df_counts = pd.read_sql_query(check_query, conn)
display(df_counts)

,tbl,cnt
0,members,150
1,hardware_pcs,30
2,game_sessions,500
3,bar_orders,350
4,tournaments,8


#### 4.2. Фильтрация и сортировка данных

In [5]:
query_42 = '''
SELECT member_id, nickname, city, loyalty_tier, balance
FROM members
WHERE city IN ('Санкт-Петербург', 'Казань') AND balance > 500
ORDER BY balance DESC, member_id ASC
LIMIT 8;
'''
df_res_42 = pd.read_sql_query(query_42, conn)
display(df_res_42)

,member_id,nickname,city,loyalty_tier,balance
0,6,Slayer_535,Санкт-Петербург,Bronze,7500.0
1,10,CyberKnight_353,Санкт-Петербург,Bronze,7500.0
2,64,Valkyrie_27,Казань,Bronze,7500.0
3,97,Rogue_918,Казань,Gold,7500.0
4,149,Rogue_509,Санкт-Петербург,Bronze,7500.0
5,5,Wolf_798,Казань,Gold,4000.0
6,23,CyberKnight_12,Казань,Gold,4000.0
7,40,Bullet_285,Казань,Bronze,4000.0


#### 4.3. Группировка и фильтрация агрегированных данных

In [6]:
query_43 = '''
SELECT game_title, COUNT(*) AS tournaments_count, ROUND(SUM(prize_fund), 2) AS total_prizes, ROUND(AVG(entry_fee), 2) AS avg_fee
FROM tournaments
GROUP BY game_title
HAVING SUM(prize_fund) >= 50000
ORDER BY total_prizes DESC;
'''
df_res_43 = pd.read_sql_query(query_43, conn)
display(df_res_43)

,game_title,tournaments_count,total_prizes,avg_fee
0,Counter-Strike 2,2,170000.0,750.0
1,Dota 2,2,105000.0,525.0


#### 4.4. Объединение таблиц

In [7]:
query_44 = '''
SELECT p.pc_id, p.pc_number, p.gpu, COUNT(s.session_id) AS sessions_cnt, ROUND(SUM(s.total_paid), 2) AS total_revenue
FROM hardware_pcs p
JOIN game_sessions s ON p.pc_id = s.pc_id
WHERE p.status = 'Active'
GROUP BY p.pc_id, p.pc_number, p.gpu
ORDER BY total_revenue DESC
LIMIT 10;
'''
df_res_44 = pd.read_sql_query(query_44, conn)
display(df_res_44)

,pc_id,pc_number,gpu,sessions_cnt,total_revenue
0,26,PC-26,RTX 4080 Super,22,19040.0
1,15,PC-15,RTX 4060,26,17525.0
2,24,PC-24,RTX 4090,20,16454.0
3,20,PC-20,RTX 4060 Ti,31,15938.0
4,17,PC-17,RTX 3060 Ti,22,11538.0
5,25,PC-25,RTX 4080 Super,13,11324.0
6,19,PC-19,RTX 3060 Ti,21,11175.0
7,21,PC-21,RTX 4060 Ti,19,10341.0
8,11,PC-11,RTX 3050,22,9848.0
9,7,PC-07,GTX 1660 Super,23,9222.0


#### 4.5. Подзапросы

In [8]:
query_45 = '''
SELECT tournament_id, tournament_name, game_title, prize_fund, entry_fee
FROM tournaments
WHERE prize_fund > (SELECT AVG(prize_fund) FROM tournaments)
ORDER BY prize_fund DESC;
'''
df_res_45 = pd.read_sql_query(query_45, conn)
display(df_res_45)

,tournament_id,tournament_name,game_title,prize_fund,entry_fee
0,106,Pro Masters Elite: CS2,Counter-Strike 2,120000.0,1000.0
1,102,Spring DOTA 2 Major,Dota 2,75000.0,700.0


### Задание 5. Поиск и исправление ошибок в SQL-запросах

#### 5.1. Выручка бара по категориям меню

**Найденные ошибки:**
1. Агрегатная функция в `WHERE`: условие `WHERE SUM(total_amount) > 10000` недопустимо, фильтрацию агрегатов следует производить в блоке `HAVING`.

In [9]:
query_51_fixed = '''
SELECT 
    category,
    COUNT(*) AS orders_cnt,
    SUM(total_amount) AS category_revenue
FROM bar_orders
GROUP BY category
HAVING SUM(total_amount) > 10000
ORDER BY category_revenue DESC;
'''
df_res_51 = pd.read_sql_query(query_51_fixed, conn)
display(df_res_51)

,category,orders_cnt,category_revenue
0,Мерч,68,102300.0
1,Напитки,131,36020.0
2,Горячее,68,27720.0
3,Снеки,83,14475.0


#### 5.2. Средняя стоимость товаров по категориям

**Найденные ошибки:**
1. Порядок секций: блок `WHERE` написан после `GROUP BY`, фильтрация групп должна производиться через `HAVING`.
2. Использование псевдонима: в фильтре указан alias `avg_unit_price` вместо выражения `AVG(item_price)`.

In [10]:
query_52_fixed = '''
SELECT 
    category,
    COUNT(*) AS orders_count,
    ROUND(AVG(item_price), 2) AS avg_unit_price
FROM bar_orders
GROUP BY category
HAVING AVG(item_price) > 100
ORDER BY avg_unit_price DESC;
'''
df_res_52 = pd.read_sql_query(query_52_fixed, conn)
display(df_res_52)

,category,orders_count,avg_unit_price
0,Мерч,68,1057.35
1,Горячее,68,285.29
2,Напитки,131,173.59
3,Снеки,83,125.60


#### 5.3. Сравнение расходов в баре со средним чеком бара

**Найденные ошибки:**
1. Ошибка ключа соединения: `m.member_id = b.order_id` связывает клиента с номером заказа вместо `b.member_id`.
2. Неполный `GROUP BY`: поле `m.nickname` выбрано в `SELECT`, но отсутствует в списке группировки.
3. Многострочный подзапрос: группировка `GROUP BY category` в подзапросе возвращает набор строк вместо одного скалярного среднего.

In [11]:
query_53_fixed = '''
SELECT 
    m.member_id,
    m.nickname,
    ROUND(AVG(b.total_amount), 2) AS avg_order_sum
FROM members m
JOIN bar_orders b ON m.member_id = b.member_id
GROUP BY m.member_id, m.nickname
HAVING AVG(b.total_amount) > (
    SELECT AVG(total_amount) 
    FROM bar_orders
)
ORDER BY avg_order_sum DESC
LIMIT 10;
'''
df_res_53 = pd.read_sql_query(query_53_fixed, conn)
display(df_res_53)

,member_id,nickname,avg_order_sum
0,135,Paladin_954,1810.0
1,113,Blade_686,1620.0
2,140,Samurai_183,1555.0
3,109,Falcon_429,1350.0
4,45,Turbo_408,1140.0
5,91,Matrix_207,1030.0
6,137,Tempest_732,970.0
7,47,Rogue_710,956.0
8,42,Ghost_633,937.5
9,71,Dragon_401,932.5
